In [1]:
import json
import requests
import numpy as np
import pandas as pd

from datetime import datetime
from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split

In [2]:
# =========================
# CONFIG
# =========================

API_BASE_URL = "http://localhost:8003"
GROUP_NUMBER = 3

MYSQL_USER = "mlops_user"
MYSQL_PASSWORD = "mlops_pass"
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DATABASE = "mlops_db"

ENGINE = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

In [3]:
from datetime import datetime
import pandas as pd


def save_batch_statistic(
    batch_number,
    feature_name,
    mean_value=None,
    std_value=None,
    new_categories_count=None
):

    row = pd.DataFrame([{
        "batch_number": batch_number,
        "feature_name": feature_name,
        "mean_value": mean_value,
        "std_value": std_value,
        "new_categories_count": new_categories_count,
        "used_for_training": False,
        "created_at": datetime.now()
    }])

    row.to_sql(
        "batch_statistics",
        ENGINE,
        if_exists="append",
        index=False
    )

In [7]:
from sqlalchemy import text


def create_tables():

    ddl_raw = """
    CREATE TABLE IF NOT EXISTS raw_data (

        raw_id BIGINT AUTO_INCREMENT PRIMARY KEY,

        batch_number INT,
        ingestion_timestamp DATETIME,

        brokered_by DOUBLE,
        status VARCHAR(100),
        price DOUBLE,
        bed DOUBLE,
        bath DOUBLE,
        acre_lot DOUBLE,
        street DOUBLE,
        city VARCHAR(255),
        state VARCHAR(255),
        zip_code VARCHAR(30),
        house_size DOUBLE,
        prev_sold_date VARCHAR(50)
    );
    """

    ddl_clean = """
    CREATE TABLE IF NOT EXISTS clean_data (

        clean_id BIGINT AUTO_INCREMENT PRIMARY KEY,

        raw_id BIGINT,
        batch_number INT,
        processed_timestamp DATETIME,

        brokered_by DOUBLE,
        status VARCHAR(100),
        price DOUBLE,
        bed DOUBLE,
        bath DOUBLE,
        acre_lot DOUBLE,
        street DOUBLE,
        city VARCHAR(255),
        state VARCHAR(255),
        zip_code VARCHAR(30),
        house_size DOUBLE,

        prev_sold_date DATE,
        years_since_last_sale DOUBLE,

        dataset_split VARCHAR(20)
    );
    """

    ddl_monitoring = """
    CREATE TABLE IF NOT EXISTS data_monitoring (

        monitoring_id BIGINT AUTO_INCREMENT PRIMARY KEY,

        batch_number INT,
        check_type VARCHAR(100),
        feature_name VARCHAR(100),
        metric_name VARCHAR(100),

        metric_value DOUBLE NULL,

        details TEXT,

        created_at DATETIME
    );
    """

    ddl_batch_statistics = """
    CREATE TABLE IF NOT EXISTS batch_statistics (

        stat_id BIGINT AUTO_INCREMENT PRIMARY KEY,

        batch_number INT NOT NULL,

        feature_name VARCHAR(100) NOT NULL,

        mean_value DOUBLE NULL,

        std_value DOUBLE NULL,

        new_categories_count INT NULL,

        used_for_training BOOLEAN DEFAULT FALSE,

        created_at DATETIME
    );
    """
    ddl_training_runs = """
    CREATE TABLE IF NOT EXISTS training_runs (
    
        run_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    
        triggering_batch INT,
    
        mlflow_run_id VARCHAR(255),
    
        candidate_rmse DOUBLE,
        candidate_mae DOUBLE,
        candidate_r2 DOUBLE,
    
        promoted BOOLEAN,
    
        created_at DATETIME
    );
    """
    with ENGINE.begin() as conn:

        conn.execute(text(ddl_raw))
        conn.execute(text(ddl_clean))
        conn.execute(text(ddl_monitoring))
        conn.execute(text(ddl_batch_statistics))
        conn.execute(text(ddl_training_runs))

    print("Tables verified.")

In [8]:
def log_monitoring(
    batch_number,
    check_type,
    feature_name,
    metric_name,
    metric_value=None,
    details=None
):

    row = pd.DataFrame([{
        "batch_number": batch_number,
        "check_type": check_type,
        "feature_name": feature_name,
        "metric_name": metric_name,
        "metric_value": metric_value,
        "details": details,
        "created_at": datetime.now()
    }])

    row.to_sql(
        "data_monitoring",
        ENGINE,
        if_exists="append",
        index=False
    )

In [9]:
def start():

    print("Checking API health...")

    response = requests.get(
        f"{API_BASE_URL}/health",
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Health check failed: {response.status_code}"
        )

    print("API health OK")

In [10]:
def fetch_and_store_raw_batch():

    response = requests.get(
        f"{API_BASE_URL}/data",
        params={"group_number": GROUP_NUMBER},
        timeout=60
    )

    response.raise_for_status()

    payload = response.json()

    batch_number = payload["batch_number"]

    df = pd.DataFrame(payload["data"])

    if df.empty:
        raise ValueError("Batch returned empty")

    df["batch_number"] = batch_number
    df["ingestion_timestamp"] = datetime.now()

    if "zip" in df.columns:
        df.rename(columns={"zip": "zip_code"}, inplace=True)

    if "zip_code" in df.columns:
        df["zip_code"] = df["zip_code"].astype(str)

    df.to_sql(
        "raw_data",
        ENGINE,
        if_exists="append",
        index=False
    )

    print(
        f"Stored {len(df)} rows for batch {batch_number}"
    )

    return batch_number

In [11]:
def validate_schema():

    query = """
    SELECT MAX(batch_number)
    FROM raw_data
    """

    batch_number = pd.read_sql(query, ENGINE).iloc[0,0]

    df = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number={batch_number}
        """,
        ENGINE
    )

    required_columns = [
        "brokered_by",
        "status",
        "price",
        "bed",
        "bath",
        "acre_lot",
        "street",
        "city",
        "state",
        "zip_code",
        "house_size",
        "prev_sold_date"
    ]

    missing = [
        c
        for c in required_columns
        if c not in df.columns
    ]

    if missing:

        log_monitoring(
            batch_number,
            "schema_validation",
            "dataset",
            "missing_columns",
            details=json.dumps(missing)
        )

        raise ValueError(
            f"Missing columns: {missing}"
        )

    log_monitoring(
        batch_number,
        "schema_validation",
        "dataset",
        "schema_ok"
    )

    print("Schema validation OK")

In [12]:
def validate_data_quality():

    batch_number = pd.read_sql(
        "SELECT MAX(batch_number) batch_number FROM raw_data",
        ENGINE
    ).iloc[0,0]

    df = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number={batch_number}
        """,
        ENGINE
    )

    duplicates = df.duplicated().sum()

    log_monitoring(
        batch_number,
        "quality_validation",
        "dataset",
        "duplicates",
        duplicates
    )

    for col in df.columns:

        nulls = int(df[col].isna().sum())

        log_monitoring(
            batch_number,
            "quality_validation",
            col,
            "null_count",
            nulls
        )

    invalid_price = (df["price"] <= 0).sum()

    log_monitoring(
        batch_number,
        "quality_validation",
        "price",
        "invalid_price",
        int(invalid_price)
    )

    print("Quality validation completed")

In [13]:
def get_latest_batch():

    return pd.read_sql(
        """
        SELECT MAX(batch_number) AS batch_number
        FROM raw_data
        """,
        ENGINE
    ).iloc[0, 0]

In [14]:
def detect_new_categories():

    categorical_cols = ["brokered_by",
                        "street",
                        "status",
                        "city",
                        "state",
                        "zip_code"
                    ]

    batch_number = pd.read_sql(
        """
        SELECT MAX(batch_number) AS batch_number
        FROM raw_data
        """,
        ENGINE
    ).iloc[0, 0]

    current = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number = {batch_number}
        """,
        ENGINE
    )

    history = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number < {batch_number}
        """,
        ENGINE
    )

    rows = []

    for col in categorical_cols:

        current_values = set(
            current[col]
            .dropna()
            .astype(str)
        )

        if history.empty:

            # primer batch:
            # todas las categorías son nuevas
            new_categories_count = len(current_values)

        else:

            historical_values = set(
                history[col]
                .dropna()
                .astype(str)
            )

            new_categories_count = len(
                current_values - historical_values
            )

        rows.append({
            "batch_number": batch_number,
            "feature_name": col,
            "mean_value": None,
            "std_value": None,
            "new_categories_count": new_categories_count,
            "used_for_training": False,
            "created_at": pd.Timestamp.now()
        })

    pd.DataFrame(rows).to_sql(
        "batch_statistics",
        ENGINE,
        if_exists="append",
        index=False
    )

    print(
        f"Stored category statistics for batch {batch_number}"
    )

In [15]:
def detect_data_drift():

    numeric_cols = ["price",
                    "bed",
                    "bath",
                    "acre_lot",
                    "house_size"
                ]

    batch_number = pd.read_sql(
        """
        SELECT MAX(batch_number) AS batch_number
        FROM raw_data
        """,
        ENGINE
    ).iloc[0, 0]

    current = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number = {batch_number}
        """,
        ENGINE
    )

    rows = []

    for col in numeric_cols:

        rows.append({
                "batch_number": batch_number,
                "feature_name": col,
                "mean_value": current[col].mean(),
                "std_value": current[col].std(),
                "new_categories_count": None,
                "used_for_training": False,
                "created_at": pd.Timestamp.now()
            })

    pd.DataFrame(rows).to_sql(
        "batch_statistics",
        ENGINE,
        if_exists="append",
        index=False
    )

    print(
        f"Stored numerical statistics for batch {batch_number}"
    )

In [16]:
def preprocess_data():

    batch_number = pd.read_sql(
        "SELECT MAX(batch_number) batch_number FROM raw_data",
        ENGINE
    ).iloc[0,0]

    raw_df = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number={batch_number}
        """,
        ENGINE
    )

    df = raw_df.copy()

    numeric_cols = [
        "brokered_by",
        "price",
        "bed",
        "bath",
        "acre_lot",
        "street",
        "house_size"
    ]

    categorical_cols = [
        "status",
        "city",
        "state",
        "zip_code"
    ]

    for col in numeric_cols:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    for col in numeric_cols:

        df[col] = df[col].fillna(
            df[col].median()
        )

    for col in categorical_cols:

        mode = (
            df[col].mode().iloc[0]
            if not df[col].mode().empty
            else "UNKNOWN"
        )

        df[col] = df[col].fillna(mode)

    df["prev_sold_date"] = pd.to_datetime(
        df["prev_sold_date"],
        errors="coerce"
    )

    today = pd.Timestamp.today()

    df["years_since_last_sale"] = (
        (today - df["prev_sold_date"]).dt.days
        / 365.25
    )

    df["years_since_last_sale"] = (
        df["years_since_last_sale"]
        .fillna(
            df["years_since_last_sale"].median()
        )
    )

    before = len(df)

    df = df.drop_duplicates()

    df = df[
        (df["price"] > 0)
        & (df["house_size"] > 0)
        & (df["bed"] >= 0)
        & (df["bath"] >= 0)
        & (df["acre_lot"] >= 0)
    ]

    removed = before - len(df)

    log_monitoring(
        batch_number,
        "preprocessing",
        "dataset",
        "rows_removed",
        removed
    )

    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42
    )

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42
    )

    train_df["dataset_split"] = "train"
    valid_df["dataset_split"] = "validation"
    test_df["dataset_split"] = "test"

    clean_df = pd.concat(
        [
            train_df,
            valid_df,
            test_df
        ],
        ignore_index=True
    )

    clean_df["processed_timestamp"] = datetime.now()

    cols = [
        "raw_id",
        "batch_number",
        "processed_timestamp",
        "brokered_by",
        "status",
        "price",
        "bed",
        "bath",
        "acre_lot",
        "street",
        "city",
        "state",
        "zip_code",
        "house_size",
        "prev_sold_date",
        "years_since_last_sale",
        "dataset_split"
    ]

    clean_df[cols].to_sql(
        "clean_data",
        ENGINE,
        if_exists="append",
        index=False
    )

    print(
        f"Inserted {len(clean_df)} rows into clean_data"
    )

In [17]:
def preprocess_data():

    batch_number = pd.read_sql(
        "SELECT MAX(batch_number) batch_number FROM raw_data",
        ENGINE
    ).iloc[0,0]

    raw_df = pd.read_sql(
        f"""
        SELECT *
        FROM raw_data
        WHERE batch_number={batch_number}
        """,
        ENGINE
    )

    df = raw_df.copy()

    numeric_cols = [
        "brokered_by",
        "price",
        "bed",
        "bath",
        "acre_lot",
        "street",
        "house_size"
    ]

    categorical_cols = [
        "status",
        "city",
        "state",
        "zip_code"
    ]

    for col in numeric_cols:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    for col in numeric_cols:

        df[col] = df[col].fillna(
            df[col].median()
        )

    for col in categorical_cols:

        mode = (
            df[col].mode().iloc[0]
            if not df[col].mode().empty
            else "UNKNOWN"
        )

        df[col] = df[col].fillna(mode)

    df["prev_sold_date"] = pd.to_datetime(
        df["prev_sold_date"],
        errors="coerce"
    )

    today = pd.Timestamp.today()

    df["years_since_last_sale"] = (
        (today - df["prev_sold_date"]).dt.days
        / 365.25
    )

    df["years_since_last_sale"] = (
        df["years_since_last_sale"]
        .fillna(
            df["years_since_last_sale"].median()
        )
    )

    before = len(df)

    df = df.drop_duplicates()

    df = df[
        (df["price"] > 0)
        & (df["house_size"] > 0)
        & (df["bed"] >= 0)
        & (df["bath"] >= 0)
        & (df["acre_lot"] >= 0)
    ]

    removed = before - len(df)

    log_monitoring(
        batch_number,
        "preprocessing",
        "dataset",
        "rows_removed",
        removed
    )

    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42
    )

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42
    )

    train_df["dataset_split"] = "train"
    valid_df["dataset_split"] = "validation"
    test_df["dataset_split"] = "test"

    clean_df = pd.concat(
        [
            train_df,
            valid_df,
            test_df
        ],
        ignore_index=True
    )

    clean_df["processed_timestamp"] = datetime.now()

    cols = [
        "raw_id",
        "batch_number",
        "processed_timestamp",
        "brokered_by",
        "status",
        "price",
        "bed",
        "bath",
        "acre_lot",
        "street",
        "city",
        "state",
        "zip_code",
        "house_size",
        "prev_sold_date",
        "years_since_last_sale",
        "dataset_split"
    ]

    clean_df[cols].to_sql(
        "clean_data",
        ENGINE,
        if_exists="append",
        index=False
    )

    print(
        f"Inserted {len(clean_df)} rows into clean_data"
    )

In [18]:
def simulate_dag_until_preprocessing():

    print("=" * 60)
    print("START DAG")
    print("=" * 60)

    start()

    batch_number = fetch_and_store_raw_batch()

    validate_schema()

    validate_data_quality()

    detect_new_categories()

    detect_data_drift()

    preprocess_data()

    print("=" * 60)
    print(
        f"PIPELINE COMPLETED - BATCH {batch_number}"
    )
    print("=" * 60)

In [25]:
if __name__ == "__main__":

    create_tables()

    simulate_dag_until_preprocessing()

Tables verified.
START DAG
Checking API health...
API health OK
Stored 73784 rows for batch 0
Schema validation OK
Quality validation completed
Stored category statistics for batch 0
Stored numerical statistics for batch 0
Inserted 221352 rows into clean_data
PIPELINE COMPLETED - BATCH 0


In [13]:
import pandas as pd
from sqlalchemy import create_engine

MYSQL_USER = "mlops_user"
MYSQL_PASSWORD = "mlops_pass"
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DATABASE = "mlops_db"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

tables = [
    "raw_data",
    "clean_data",
    "data_monitoring",
    "batch_statistics"
]

for table in tables:

    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    count_query = f"SELECT COUNT(*) AS total_rows FROM {table}"
    total_rows = pd.read_sql(count_query, engine)["total_rows"].iloc[0]

    print(f"\nTotal rows: {total_rows}")

    if total_rows > 0:

        sample = pd.read_sql(
            f"SELECT * FROM {table} LIMIT 40",
            engine
        )

        print("\nColumns:")
        print(list(sample.columns))

        print("\nSample:")
        print(sample)

    else:
        print("Table is empty")


TABLE: raw_data

Total rows: 168335

Columns:
['raw_id', 'batch_number', 'ingestion_timestamp', 'brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']

Sample:
    raw_id  batch_number ingestion_timestamp  brokered_by    status     price  \
0        1             0 2026-06-03 07:18:40     101640.0  for_sale  289900.0   
1        2             0 2026-06-03 07:18:40     107951.0  for_sale  299900.0   
2        3             0 2026-06-03 07:18:40      80935.0  for_sale  299000.0   
3        4             0 2026-06-03 07:18:40      33714.0  for_sale  221000.0   
4        5             0 2026-06-03 07:18:40      29997.0  for_sale  175000.0   
5        6             0 2026-06-03 07:18:40      10571.0  for_sale  140000.0   
6        7             0 2026-06-03 07:18:40      51868.0  for_sale  270000.0   
7        8             0 2026-06-03 07:18:40      33503.0  for_sale  294900.0   
8        9             0 2026-06-

In [18]:
query = """
SELECT *
FROM batch_statistics
WHERE feature_name = 'price'
"""

df = pd.read_sql(query, engine)

print(df.to_string())

    batch_number feature_name          mean_value          std_value new_categories_count  used_for_training          created_at
0              0        price   219185.5700964979  64932.62105938571                 None                  1 2026-06-03 04:50:15
1              1        price  179512.42678554432  70795.19540879992                 None                  1 2026-06-03 04:50:39
2              3        price  236809.57731196054  53602.62689040461                 None                  1 2026-06-03 05:11:23
3              3        price  236809.57731196054  53602.62689040461                 None                  1 2026-06-03 05:45:02
4              0        price   219185.5700964979  64932.62105938571                 None                  1 2026-06-03 05:55:06
5              1        price  179512.42678554432  70795.19540879992                 None                  1 2026-06-03 06:06:13
6              1        price  179512.42678554432  70795.19540879992                 None        

In [19]:
print(df.dtypes)

batch_number                     int64
feature_name                    object
mean_value                      object
std_value                       object
new_categories_count            object
used_for_training                int64
created_at              datetime64[ns]
dtype: object


In [17]:
bad_rows

,batch_number,feature_name,mean_value,std_value,new_categories_count,used_for_training,created_at,mean_value_numeric
0,0,brokered_by,None,None,20856.0,1,2026-06-03 04:50:08,NaN
1,0,street,None,None,73400.0,1,2026-06-03 04:50:08,NaN
2,0,status,None,None,1.0,1,2026-06-03 04:50:08,NaN
3,0,city,None,None,7747.0,1,2026-06-03 04:50:08,NaN
4,0,state,None,None,50.0,1,2026-06-03 04:50:08,NaN
...,...,...,...,...,...,...,...,...
133,1,street,None,None,93650.0,0,2026-06-03 07:19:28,NaN
134,1,status,None,None,0.0,0,2026-06-03 07:19:28,NaN
135,1,city,None,None,2400.0,0,2026-06-03 07:19:28,NaN
136,1,state,None,None,1.0,0,2026-06-03 07:19:28,NaN


In [12]:
from sqlalchemy import create_engine, text

MYSQL_USER = "mlops_user"
MYSQL_PASSWORD = "mlops_pass"
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DATABASE = "mlops_db"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)


tables = [
    "raw_data",
    "clean_data",
    "data_monitoring",
    "batch_statistics"
    "training_runs"
    "inference_logs"
]

with engine.begin() as conn:
    for table in tables:
        conn.execute(text(f"DROP TABLE IF EXISTS `{table}`"))
        print(f"Tabla eliminada: {table}")

print("Proceso finalizado.")

Tabla eliminada: raw_data
Tabla eliminada: clean_data
Tabla eliminada: data_monitoring
Tabla eliminada: batch_statisticstraining_runsinference_logs
Proceso finalizado.


In [10]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = client.get_experiment_by_name(
    "real_estate_price_prediction"
)

if experiment:
    client.restore_experiment(
        experiment.experiment_id
    )

    print(
        f"Experiment restored: {experiment.experiment_id}"
    )
else:
    print("Experiment not found")

Experiment not found
